# Taller interactivo: Transformacion de una casa en R^3

En este taller veremos como una matriz de transformacion lineal en $\mathbb{R}^3$ cambia la forma y la orientacion de una casa 3D.

Objetivos:
- Representar una casa en 3D mediante vertices y aristas.
- Aplicar una matriz $A\in\mathbb{R}^{3\times 3}$ a todos los puntos.
- Explorar reflexiones hacia distintos octantes usando matrices diagonales con signos.
- Manipular la matriz y observar el efecto grafico en tiempo real.

## Recordatorio teorico
Si $v$ es un punto de la casa, su transformacion es:
$$
T(v)=Av
$$
donde $A$ es una matriz $3\times 3$.

Reflexiones tipicas:
- Respecto al plano $yz$: $\mathrm{diag}(-1,1,1)$
- Respecto al plano $xz$: $\mathrm{diag}(1,-1,1)$
- Respecto al plano $xy$: $\mathrm{diag}(1,1,-1)$
- Respecto al origen: $-I$

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

%matplotlib inline

In [2]:
# Vertices de una casa en 3D
V = np.array([
    [-1.0, -1.0, 0.0],  # 0 base
    [ 1.0, -1.0, 0.0],  # 1
    [ 1.0,  1.0, 0.0],  # 2
    [-1.0,  1.0, 0.0],  # 3
    [-1.0, -1.0, 1.0],  # 4 techo base
    [ 1.0, -1.0, 1.0],  # 5
    [ 1.0,  1.0, 1.0],  # 6
    [-1.0,  1.0, 1.0],  # 7
    [ 0.0, -1.0, 1.8],  # 8 cumbrera frente
    [ 0.0,  1.0, 1.8],  # 9 cumbrera fondo
])

# Aristas (pares de indices de vertices)
E = [
    (0, 1), (1, 2), (2, 3), (3, 0),
    (0, 4), (1, 5), (2, 6), (3, 7),
    (4, 5), (5, 6), (6, 7), (7, 4),
    (4, 8), (5, 8), (7, 9), (6, 9),
    (8, 9)
]

def aplicar_transformacion(A, puntos):
    return (A @ puntos.T).T

def dibujar_casa(ax, puntos, color='tab:blue', titulo='Casa'):
    for i, j in E:
        x = [puntos[i, 0], puntos[j, 0]]
        y = [puntos[i, 1], puntos[j, 1]]
        z = [puntos[i, 2], puntos[j, 2]]
        ax.plot(x, y, z, color=color, linewidth=2)

    ax.scatter(puntos[:, 0], puntos[:, 1], puntos[:, 2], color=color, s=25)
    ax.set_title(titulo)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_zlabel('z')
    ax.set_box_aspect((1, 1, 1))

In [3]:
# Widgets para editar la matriz A
a11 = widgets.FloatText(value=1.0, description='a11')
a12 = widgets.FloatText(value=0.0, description='a12')
a13 = widgets.FloatText(value=0.0, description='a13')
a21 = widgets.FloatText(value=0.0, description='a21')
a22 = widgets.FloatText(value=1.0, description='a22')
a23 = widgets.FloatText(value=0.0, description='a23')
a31 = widgets.FloatText(value=0.0, description='a31')
a32 = widgets.FloatText(value=0.0, description='a32')
a33 = widgets.FloatText(value=1.0, description='a33')

def matriz_actual():
    return np.array([
        [a11.value, a12.value, a13.value],
        [a21.value, a22.value, a23.value],
        [a31.value, a32.value, a33.value],
    ], dtype=float)

# Octantes por signos (x, y, z)
octantes = {
    '(+,+,+)': ( 1,  1,  1),
    '(-,+,+)': (-1,  1,  1),
    '(+,-,+)': ( 1, -1,  1),
    '(+,+,-)': ( 1,  1, -1),
    '(-,-,+)': (-1, -1,  1),
    '(-,+,-)': (-1,  1, -1),
    '(+,-,-)': ( 1, -1, -1),
    '(-,-,-)': (-1, -1, -1),
}

selector_octante = widgets.Dropdown(
    options=list(octantes.keys()),
    value='(+,+,+)',
    description='Octante:'
)

btn_reflexion = widgets.Button(description='Cargar reflexion al octante', button_style='info')
btn_aplicar = widgets.Button(description='Aplicar y graficar', button_style='success')
salida = widgets.Output()

def cargar_reflexion(_):
    sx, sy, sz = octantes[selector_octante.value]
    a11.value, a12.value, a13.value = sx, 0.0, 0.0
    a21.value, a22.value, a23.value = 0.0, sy, 0.0
    a31.value, a32.value, a33.value = 0.0, 0.0, sz

def graficar(_=None):
    A = matriz_actual()
    VT = aplicar_transformacion(A, V)
    detA = np.linalg.det(A)

    with salida:
        clear_output(wait=True)
        fig = plt.figure(figsize=(12, 5))

        ax1 = fig.add_subplot(121, projection='3d')
        dibujar_casa(ax1, V, color='tab:blue', titulo='Casa original')

        ax2 = fig.add_subplot(122, projection='3d')
        dibujar_casa(ax2, VT, color='tab:red', titulo='Casa transformada')

        # Limites comunes para comparar mejor
        all_pts = np.vstack([V, VT])
        lim = np.max(np.abs(all_pts)) + 0.5
        for ax in (ax1, ax2):
            ax.set_xlim([-lim, lim])
            ax.set_ylim([-lim, lim])
            ax.set_zlim([-lim, lim])
            ax.view_init(elev=20, azim=35)
            ax.grid(True, alpha=0.35)

        plt.tight_layout()
        plt.show()

        print('Matriz A usada:')
        print(A)
        print(f'det(A) = {detA:.3f}')
        if detA < 0:
            print('Interpretacion: hay cambio de orientacion (incluye una reflexion).')
        elif detA == 0:
            print('Interpretacion: la transformacion colapsa dimensiones (no es invertible).')
        else:
            print('Interpretacion: conserva orientacion (sin reflexion pura).')

btn_reflexion.on_click(cargar_reflexion)
btn_aplicar.on_click(graficar)

fila1 = widgets.HBox([a11, a12, a13])
fila2 = widgets.HBox([a21, a22, a23])
fila3 = widgets.HBox([a31, a32, a33])
controles = widgets.VBox([
    widgets.HTML('<b>Matriz de transformacion A</b>'),
    fila1, fila2, fila3,
    selector_octante,
    widgets.HBox([btn_reflexion, btn_aplicar])
])

display(controles, salida)

# Grafica inicial
graficar()

Output()

## Actividades propuestas
1. Carga la reflexion hacia cada octante y describe que coordenadas cambian de signo.
2. Compara $\det(A)$ para diferentes reflexiones. Que patron observas?
3. Prueba matrices no diagonales (por ejemplo cizalla o rotacion aproximada) y describe el efecto.
4. Encuentra una matriz que lleve un vertice especifico de la casa al octante $(-,-,+)$ sin colapsar la figura.
5. Investiga: cuando la transformacion es invertible y como recuperar la casa original con $A^{-1}$.